# Detect SWR & Spdl

## 0. Installation et mises à jour (à lancer à chaque séance)

Prérequis : travailler sur le cluster (JupyterHub ou nœud interactif) avec le kernel `.venv` de ce repo, et avoir un accès SSH au GitLab du labo depuis le cluster (`ssh -T git@10.69.168.17` doit répondre `Welcome to GitLab`).

1. Exécuter la cellule ci-dessous. Elle installe dans l'environnement du kernel la dernière version corrective de la série indiquée (`SERIE`), seulement si elle est plus récente que celle installée. Sinon, elle dit que tout est à jour, en une seconde.
2. Si elle a installé quelque chose : **redémarrer le kernel** (bouton *Restart*), puis continuer.

Les détections envoyées au cluster utilisent ce même environnement : la cellule n'installe rien tant que des jobs tournent encore.

Si elle annonce une nouvelle **série** (par exemple 0.20), ne pas la changer soi-même : elle peut demander d'adapter ce notebook, voir avec Mélodie.


In [1]:
import os
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

SERIE = "0.19"   # série suivie : seules ses versions correctives (0.19.x) s'installent automatiquement
GITLAB = "git@10.69.168.17"
PACKAGE = "haylab-tools[swr,uds,cluster] @ git+ssh://" + GITLAB + "/haygroup/haylab-tools.git@v{}"

print("Environnement du kernel :", sys.prefix)   # doit être le .venv de ce repo
# accepte la clé du serveur GitLab à la première connexion (%pip ne peut pas répondre "yes"), refuse toujours une clé qui changerait
os.environ["GIT_SSH_COMMAND"] = "ssh -o StrictHostKeyChecking=accept-new"

def as_tuple(v):
    return tuple(int(x) for x in v.split("."))

tags = subprocess.run(["git", "ls-remote", "--tags", "--refs", f"ssh://{GITLAB}/haygroup/haylab-tools.git"],
                      capture_output=True, text=True, check=True).stdout
versions = sorted(as_tuple(v) for v in re.findall(r"refs/tags/v(\d+\.\d+\.\d+)$", tags, re.M))
in_serie = [v for v in versions if v[:2] == as_tuple(SERIE)]
if not in_serie:
    raise ValueError(f"Aucune version {SERIE}.x sur GitLab : vérifier SERIE, ou voir avec Mélodie.")
latest = ".".join(map(str, max(in_serie)))
if any(v[:2] > as_tuple(SERIE) for v in versions):
    print(f"Une nouvelle série existe ({'.'.join(map(str, versions[-1][:2]))}) : voir avec Mélodie pour adapter ce notebook.")

try:
    installed = version("haylab-tools")
except PackageNotFoundError:
    installed = None
# ses jobs SLURM, sauf celui qui fait tourner ce notebook s'il y en a un
running = set(subprocess.run(["squeue", "-h", "-u", os.environ["USER"], "-o", "%i"],
                             capture_output=True, text=True).stdout.split()) - {os.environ.get("SLURM_JOB_ID")}
if installed == latest:
    print(f"haylab-tools {installed}, à jour.")
elif running:
    print(f"haylab-tools {installed} -> {latest} reporté : des jobs tournent encore avec cet environnement "
          f"({', '.join(sorted(running))}). Relancer cette cellule une fois finis.")
else:
    print(f"haylab-tools {installed} -> {latest}")
    spec = PACKAGE.format(latest)
    %pip install -q "{spec}"
    print(spec)
    print(">>> Redémarrer le kernel avant de continuer.")


Environnement du kernel : /home/thea.michel/HayLabAnalysis/.venv
haylab-tools 0.19.0, à jour.


In [2]:
import os
import numpy as np
import pandas as pd
import pynapple as nap
from pathlib import Path
import warnings
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, TimeFreqViewer
from ephyviewer import InMemoryAnalogSignalSource
import json
import IPython
from matplotlib import cm
from matplotlib.colors import to_hex
import ast
from ephyviewer import AnalogSignalSourceWithScatter
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import sys
from haylab_analysis.ephys import collect_swr, collect_uds, detect_swr, detect_uds
warnings.filterwarnings("ignore")

# détections sur le cluster, avec le même environnement que ce notebook
HOST = "CRNLcluster"
VENV = f"source {sys.prefix}/bin/activate"

#Load LFP coordinates 
Channels = f'{os.getcwd()}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0)

Choose file

In [3]:
try: # tries to retrieve dpath either from a previous run or from a previous notebook
    %store -r dpath
    #dpath=dpath.parent
except:
    print("the path was not defined in store")
    dpath = "//10.69.168.1/crnldata/waking/audrey_hay/L1imaging/Analysed2025_AB/"


fc1 = FileChooser(dpath,select_default=True, show_only_dirs = True, title = "<b>Go inside the folder containing the LFP raw file</b>", layout=widgets.Layout(width='100%'))
display(fc1)

# Sample callback function
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 

# Register callback function
fc1.register_callback(update_my_folder)

FileChooser(path='/crnldata/forgetting/Théa/MiniscopeOE_analysis/Allocentrick_task/1_VigSt_2026-10-23_17_19_36…

Load file

In [4]:
folder_base = Path(dpath) #Path(dpath).parent
print(folder_base)        
      
#Load signals

LFPfile = Path(f'{folder_base}/DataFrame_rawdataDS.pkl')
LFPs_df = pd.read_pickle(LFPfile)
samplerate = 1000 
numchannel = LFPs_df.shape[1]
rec_ch_list = LFPs_df.columns.values
# Load LFPs timestamps 
for file_pathTS in folder_base.parent.parent.glob('**/continuous/*/timeStampsDS.npy'):
    print('LFPs timestamps file found')
    LFPtimestamps = np.load(file_pathTS)  
print(round(LFPs_df.shape[0]/samplerate/60), 'min of recording')

# Identify mouse & choose threshold for detection
mouse = []
pos_mice = []
for mouse_name in all_LFPcoordinates.index:
    if mouse_name in LFPfile.__str__():
        mouse.append(mouse_name)
        pos_mice.append(LFPfile.__str__().find(mouse_name)) 
mouse = [x for _, x in sorted(zip(pos_mice, mouse))] # sort mouse in the same order as they appear in the path
mouse=mouse[0]   

# Identify electrodes & create differential LFPs
all_LFPcoordinates = all_LFPcoordinates.astype(str)
for region in all_LFPcoordinates.loc[mouse].index:
    locals()[region] = []
    locals()[f'{region}_ch'] = []
ID=0
RecordedArea=[]
combined0=[]
rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
for rec_ch in rec_ch_list_mouse:
    for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
        region = all_LFPcoordinates.loc[mouse].index[idx]
        if LFPcoord_str != 'nan':
            LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region 
            num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
            if len(num_ch) > 0:
                region = all_LFPcoordinates.loc[mouse].index[idx]
                LFP = locals()[region]
                LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                locals()[region] = LFP
                locals()[f'{region}_ch'].append(rec_ch)
                
                break
            continue    
for region in all_LFPcoordinates.loc[mouse].index:
    LFP = locals()[region]
    LFP_ch = locals()[f'{region}_ch']
    if len(LFP) > 0:
        RecordedArea.append(region) 
        combined0 = zscore(LFP[:,np.newaxis]) if len(combined0) == 0 else np.append(combined0, zscore(LFP[:,np.newaxis]), axis=1)

    
# Load Sleep Scoring    
ScoringFile= folder_base / f'EphyViewer_ManualScor5s_6Stages.csv'
SleepScored = pd.read_csv(ScoringFile)
SleepScored['label']= SleepScored['label'].str.extract(r'(\d+)', expand=False)
SleepScoredTS=np.array(SleepScored['label'])
scale_factor=samplerate/0.2  #cause scoring was done in 5 seconds bin, ie 0.2 Hz              
SleepScoredTS_upscaled0 = np.repeat(SleepScoredTS, scale_factor, axis=0)
SleepScoredTS_upscaled0=SleepScoredTS_upscaled0.astype(int)
SleepScoredTS_upscaled=SleepScoredTS_upscaled0.copy()

#SleepScoredTS_upscaled[SleepScoredTS_upscaled==2]=1 # Transform QW in AW
SleepScoredTS_upscaled[SleepScoredTS_upscaled==4]=1 # Transform REM in AW
SleepScoredTS_upscaled[SleepScoredTS_upscaled==5]=1 # Transform IS in AW
SleepScoredTS_upscaled[SleepScoredTS_upscaled==6]=1 # Transform undefined in AW

if len(CA1) > len(SleepScoredTS_upscaled):
    SleepScoredTS_upscaled = np.pad(SleepScoredTS_upscaled, (0, (len(CA1) - len(SleepScoredTS_upscaled))), constant_values=SleepScoredTS_upscaled[-1])
elif len(CA1) < len(SleepScoredTS_upscaled):
    SleepScoredTS_upscaled=SleepScoredTS_upscaled[:len(CA1)]

notNREMBool = SleepScoredTS_upscaled==1 


RSC=zscore(np.array(RSC))#-np.mean(np.array(CA1)))#/np.mean(np.array(CA1))
CA1=zscore(np.array(CA1))#-np.mean(np.array(CA1)))#/np.mean(np.array(CA1))
S1=zscore(np.array(S1))#-np.mean(np.array(S1)))#/np.mean(np.array(S1))
PFC=zscore(np.array(PFC))#-np.mean(np.array(PFC)))#/np.mean(np.array(PFC))

CA1[notNREMBool] = 0
S1[notNREMBool] = 0
PFC[notNREMBool] = 0

try: 
    RSC[notNREMBool] = 0 
except: 
    print('!!!! No RSC LFP !!!!')
    thRSC= None
    emp=[]
    All_SWR = pd.DataFrame(emp, columns = ['start time', 'end time', 'Duration', 'toKeep'])
    filename = folder_base / f'SWRRSC_finedetection.csv'
    All_SWR.to_csv(filename)


/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NB/Allocentric_task/Training/2026_02_16/SleepBefore/11-35-07_11-50-22/OpenEphys
15 min of recording


SWR CA1

In [5]:
SWR_PARAMS = {"threshold": 0.5, "min_event_duration_ms": 20.0, "max_event_duration_ms": 200.0}

csv = folder_base / f"SWRs_{mouse}_ml_CA1.csv"
if csv.exists():
    print(f"Déjà détecté : {csv.name} relu. Le supprimer pour relancer la détection.")
else:
    job = detect_swr(np.asarray(CA1), np.asarray(EMG) if len(EMG) > 0 else None,
                     sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                     mouse_id=mouse, session_id="CA1", output_suffix="_ml_CA1",
                     host=HOST, remote_activate_cmd=VENV, **SWR_PARAMS)
    collect_swr(job, timeout=None)   # attend la fin du job sur le cluster
    
All_SWR = pd.read_csv(csv)

mingap=1 #in seconds
prevSwr=[]
for swr in All_SWR.index :
    start=All_SWR.loc[swr, "start time"]
    previousend=All_SWR.loc[prevSwr, "end time"] if prevSwr else start-mingap*1000
    prevSwr=swr

    if start - previousend >= mingap*1000:
        All_SWR.loc[swr, "Overlapping_Baseline"] = 'False'
    else:
        All_SWR.loc[swr, "Overlapping_Baseline"] = 'True'

# Store the results in All_SWR_prop pd dataframe and save as pkl/csv for post processing.
filename = folder_base / f'SWRCA1_MLdetection.csv'
All_SWR.to_csv(filename)

print(len(All_SWR), "SWR detected in CA1 using ML")


✓ Submitted 1 task(s) as SLURM job 86008  (log: /home/thea.michel/.haylab_tools/tasks/runtime_logs/orchestrator/orchestrator-86008.out)
109 SWR detected in CA1 using ML


SWR RSC

In [6]:
SWR_PARAMS = {"threshold": 0.5, "min_event_duration_ms": 10.0, "max_event_duration_ms": 200.0}

csv = folder_base / f"SWRs_{mouse}_ml_RSC.csv"
if csv.exists():
    print(f"Déjà détecté : {csv.name} relu. Le supprimer pour relancer la détection.")
else:
    job = detect_swr(np.asarray(RSC), np.asarray(EMG) if len(EMG) > 0 else None,
                     sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                     mouse_id=mouse, session_id="RSC", output_suffix="_ml_RSC",
                     host=HOST, remote_activate_cmd=VENV, **SWR_PARAMS)
    collect_swr(job, timeout=None)   # attend la fin du job sur le cluster
All_SWR = pd.read_csv(csv)

mingap=1 #in seconds
prevSwr=[]
for swr in All_SWR.index :
    start=All_SWR.loc[swr, "start time"]
    previousend=All_SWR.loc[prevSwr, "end time"] if prevSwr else start-mingap*1000
    prevSwr=swr

    if start - previousend >= mingap*1000:
        All_SWR.loc[swr, "Overlapping_Baseline"] = 'False'
    else:
        All_SWR.loc[swr, "Overlapping_Baseline"] = 'True'

# Store the results in All_SWR_prop pd dataframe and save as pkl/csv for post processing.
filename = folder_base / f'SWRRSC_MLdetection.csv'
All_SWR.to_csv(filename)

print(len(All_SWR), "SWR detected in RSC using ML")


✓ Submitted 1 task(s) as SLURM job 86012  (log: /home/thea.michel/.haylab_tools/tasks/runtime_logs/orchestrator/orchestrator-86012.out)
48 SWR detected in RSC using ML


UDS (down-states)

Le modèle a été entraîné sur un LFP bipolaire de M1, avec l'EMG. Ces souris n'ont pas de M1 : il est appliqué ici à la région choisie dans `UDS_REGION`, à valider sur quelques sessions avant de s'y fier.


In [7]:
UDS_REGION = "S1"   # ou "S1"

csv = folder_base / f"UDS_{mouse}_ml_{UDS_REGION}.csv"
if csv.exists():
    print(f"Déjà détecté : {csv.name} relu. Le supprimer pour relancer la détection.")
else:
    job = detect_uds(np.asarray(locals()[UDS_REGION]), np.asarray(EMG),
                     sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                     mouse_id=mouse, session_id=UDS_REGION, output_suffix=f"_ml_{UDS_REGION}",
                     host=HOST, remote_activate_cmd=VENV)
    collect_uds(job, timeout=None)   # attend la fin du job sur le cluster
All_UDS = pd.read_csv(csv)


mingap=1 #in seconds
prevDS=[]
for ds in All_UDS.index :
    start=All_UDS.loc[ds, "start time"]
    previousend=All_UDS.loc[prevSwr, "end time"] if prevDS else start-mingap*1000
    prevDS=ds

    if start - previousend >= mingap*1000:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'False'
    else:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'True'

# Store the results in All_SWR_prop pd dataframe and save as pkl/csv for post processing.
filename = folder_base / f'UDS{UDS_REGION}_MLdetection.csv'
All_UDS.to_csv(filename)

print(len(All_UDS), f"down-states detected in {UDS_REGION} using ML")

✓ Submitted 1 task(s) as SLURM job 86016  (log: /home/thea.michel/.haylab_tools/tasks/runtime_logs/orchestrator/orchestrator-86016.out)
1466 down-states detected in S1 using ML


In [8]:
UDS_REGION = "PFC"   # ou "S1"

csv = folder_base / f"UDS_{mouse}_ml_{UDS_REGION}.csv"
if csv.exists():
    print(f"Déjà détecté : {csv.name} relu. Le supprimer pour relancer la détection.")
else:
    job = detect_uds(np.asarray(locals()[UDS_REGION]), np.asarray(EMG),
                     sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                     mouse_id=mouse, session_id=UDS_REGION, output_suffix=f"_ml_{UDS_REGION}",
                     host=HOST, remote_activate_cmd=VENV)
    collect_uds(job, timeout=None)   # attend la fin du job sur le cluster
All_UDS = pd.read_csv(csv)


mingap=1 #in seconds
prevDS=[]
for ds in All_UDS.index :
    start=All_UDS.loc[ds, "start time"]
    previousend=All_UDS.loc[prevSwr, "end time"] if prevDS else start-mingap*1000
    prevDS=ds

    if start - previousend >= mingap*1000:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'False'
    else:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'True'

# Store the results in All_SWR_prop pd dataframe and save as pkl/csv for post processing.
filename = folder_base / f'UDS{UDS_REGION}_MLdetection.csv'
All_UDS.to_csv(filename)

print(len(All_UDS), f"down-states detected in {UDS_REGION} using ML")

✓ Submitted 1 task(s) as SLURM job 86020  (log: /home/thea.michel/.haylab_tools/tasks/runtime_logs/orchestrator/orchestrator-86020.out)
348 down-states detected in PFC using ML


In [9]:
UDS_REGION = "RSC"   # ou "S1"

csv = folder_base / f"UDS_{mouse}_ml_{UDS_REGION}.csv"
if csv.exists():
    print(f"Déjà détecté : {csv.name} relu. Le supprimer pour relancer la détection.")
else:
    job = detect_uds(np.asarray(locals()[UDS_REGION]), np.asarray(EMG),
                     sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                     mouse_id=mouse, session_id=UDS_REGION, output_suffix=f"_ml_{UDS_REGION}",
                     host=HOST, remote_activate_cmd=VENV)
    collect_uds(job, timeout=None)   # attend la fin du job sur le cluster
All_UDS = pd.read_csv(csv)


mingap=1 #in seconds
prevDS=[]
for ds in All_UDS.index :
    start=All_UDS.loc[ds, "start time"]
    previousend=All_UDS.loc[prevSwr, "end time"] if prevDS else start-mingap*1000
    prevDS=ds

    if start - previousend >= mingap*1000:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'False'
    else:
        All_UDS.loc[ds, "Overlapping_Baseline"] = 'True'

# Store the results in All_SWR_prop pd dataframe and save as pkl/csv for post processing.
filename = folder_base / f'UDS{UDS_REGION}_MLdetection.csv'
All_UDS.to_csv(filename)

print(len(All_UDS), f"down-states detected in {UDS_REGION} using ML")

✓ Submitted 1 task(s) as SLURM job 86024  (log: /home/thea.michel/.haylab_tools/tasks/runtime_logs/orchestrator/orchestrator-86024.out)
114 down-states detected in RSC using ML


# Auto processing

In [ ]:
import os
import numpy as np
import pandas as pd
import pynapple as nap
from pathlib import Path
import warnings
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, TimeFreqViewer
from ephyviewer import InMemoryAnalogSignalSource
import json
import IPython
from matplotlib import cm
from matplotlib.colors import to_hex
import ast
from ephyviewer import AnalogSignalSourceWithScatter
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import sys
from haylab_analysis.ephys import collect_swr, collect_uds, detect_swr, detect_uds
warnings.filterwarnings("ignore")

# détections sur le cluster, avec le même environnement que ce notebook
HOST = "CRNLcluster"
VENV = f"source {sys.prefix}/bin/activate"

#Load LFP coordinates 
Channels = f'{os.getcwd()}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0)

def find_label(event_time):
    for _, row in SleepScored.iterrows():
        if row['time'] <= event_time/1000 < row['end_time']:
            return row['label']
    return None  # if no matching interval


In [ ]:
SWR_PARAMS = {"CA1": {"threshold": 0.5, "min_event_duration_ms": 20.0, "max_event_duration_ms": 200.0},
              "RSC": {"threshold": 0.1, "min_event_duration_ms": 5.0, "max_event_duration_ms": 200.0}}

dpath = Path("/crnldata/forgetting/Aurelie/MiniscopeOE_data/L2_3_mice/")
sessions = []   # (folder_base, mouse, ScoringFile), pour la cellule suivante
jobs = []       # détections envoyées au cluster, récupérées dans la cellule suivante
for ScoringFile in dpath.rglob("*EphyViewer_ManualScor5s_6Stages.csv"):
    if 'SleepAfter' in str(ScoringFile) and 'Exploration_task' in str(ScoringFile):

        print(f"Processing file: {ScoringFile}")
        mouse = ScoringFile.parents[5].name

        # Load signals
        folder_base= ScoringFile.parent
        LFPfile = f"{folder_base}/DataFrame_rawdataDS.pkl"
        LFPs_df = pd.read_pickle(LFPfile)
        samplerate = 1000 
        numchannel = LFPs_df.shape[1]
        rec_ch_list = LFPs_df.columns.values
        print(round(LFPs_df.shape[0]/samplerate/60), 'min of recording')

        # Identify mouse & choose threshold for detection
        mouse = []
        pos_mice = []
        for mouse_name in all_LFPcoordinates.index:
            if mouse_name in LFPfile.__str__():
                mouse.append(mouse_name)
                pos_mice.append(LFPfile.__str__().find(mouse_name)) 
        mouse = [x for _, x in sorted(zip(pos_mice, mouse))] # sort mouse in the same order as they appear in the path
        mouse=mouse[0]   

        # Identify electrodes & create differential LFPs
        all_LFPcoordinates = all_LFPcoordinates.astype(str)
        for region in all_LFPcoordinates.loc[mouse].index:
            locals()[region] = []
            locals()[f'{region}_ch'] = []
        ID=0
        RecordedArea=[]
        combined0=[]
        rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
        for rec_ch in rec_ch_list_mouse:
            for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
                region = all_LFPcoordinates.loc[mouse].index[idx]
                if LFPcoord_str != 'nan':
                    LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region 
                    num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
                    if len(num_ch) > 0:
                        region = all_LFPcoordinates.loc[mouse].index[idx]
                        LFP = locals()[region]
                        LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                        locals()[region] = LFP
                        locals()[f'{region}_ch'].append(rec_ch)
                        
                        break
                    continue    
        for region in all_LFPcoordinates.loc[mouse].index:
            LFP = locals()[region]
            LFP_ch = locals()[f'{region}_ch']
            if len(LFP) > 0:
                RecordedArea.append(region) 
                combined0 = zscore(LFP[:,np.newaxis]) if len(combined0) == 0 else np.append(combined0, zscore(LFP[:,np.newaxis]), axis=1)

            
        # Load Sleep Scoring    
        SleepScored = pd.read_csv(ScoringFile)
        SleepScored['label']= SleepScored['label'].str.extract(r'(\d+)', expand=False)
        SleepScored['end_time'] = SleepScored['time'] + SleepScored['duration']
        SleepScoredTS=np.array(SleepScored['label'])
        scale_factor=samplerate/0.2  #cause scoring was done in 5 seconds bin, ie 0.2 Hz              
        SleepScoredTS_upscaled0 = np.repeat(SleepScoredTS, scale_factor, axis=0)
        SleepScoredTS_upscaled0=SleepScoredTS_upscaled0.astype(int)
        SleepScoredTS_upscaled=SleepScoredTS_upscaled0.copy()

        SleepScoredTS_upscaled[SleepScoredTS_upscaled==6]=1 # Transform undefined in AW
        SleepScoredTS_upscaled[SleepScoredTS_upscaled==4]=1 # Transform REM in AW
        SleepScoredTS_upscaled[SleepScoredTS_upscaled==5]=1 # Transform IS in AW

        if len(CA1) > len(SleepScoredTS_upscaled):
            SleepScoredTS_upscaled = np.pad(SleepScoredTS_upscaled, (0, (len(CA1) - len(SleepScoredTS_upscaled))), constant_values=SleepScoredTS_upscaled[-1])
        elif len(CA1) < len(SleepScoredTS_upscaled):
            SleepScoredTS_upscaled=SleepScoredTS_upscaled[:len(CA1)]
        wakeBool = SleepScoredTS_upscaled==1 
        RSC=zscore(np.array(RSC))#-np.mean(np.array(CA1)))#/np.mean(np.array(CA1))
        CA1=zscore(np.array(CA1))#-np.mean(np.array(CA1)))#/np.mean(np.array(CA1))
        S1=zscore(np.array(S1))#-np.mean(np.array(S1)))#/np.mean(np.array(S1))
        PFC=zscore(np.array(PFC))#-np.mean(np.array(PFC)))#/np.mean(np.array(PFC))
        CA1[wakeBool] = 0
        S1[wakeBool] = 0
        PFC[wakeBool] = 0
        if len(RSC) > 0:
            RSC[wakeBool] = 0

        # Send the SWR detections (RSC, CA1) to the cluster
        sessions.append((folder_base, mouse, ScoringFile))
        for region, LFP in (("RSC", RSC), ("CA1", CA1)):
            if len(LFP) == 0:
                print(f'!!!! No {region} LFP !!!!')
            elif (folder_base / f"SWRs_{mouse}_ml_{region}.csv").exists():
                print(f"{region} : déjà détecté, résultats relus dans la cellule suivante")
            else:
                jobs.append(detect_swr(np.asarray(LFP), np.asarray(EMG) if len(EMG) > 0 else None,
                                       sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                                       mouse_id=mouse, session_id=region, output_suffix=f"_ml_{region}",
                                       host=HOST, remote_activate_cmd=VENV, **SWR_PARAMS[region]))
        for region, LFP in (("RSC", RSC), ("S1", S1), ("PFC", PFC)):
            if len(LFP) == 0:
                print(f'!!!! No {region} LFP !!!!')
            elif (folder_base / f"UDSs_{mouse}_ml_{region}.csv").exists():
                print(f"{region} : déjà détecté, résultats relus dans la cellule suivante")
            else:
                jobs.append(detect_uds(np.asarray(LFP), np.asarray(EMG),
                                        sampling_rate_hz=float(samplerate), output_dir=str(folder_base),
                                        mouse_id=mouse, session_id=region, output_suffix=f"_ml_{region}",
                                        host=HOST, remote_activate_cmd=VENV))

In [ ]:
for job in jobs:
    collect_swr(job, timeout=None)   # attend la fin de chaque job ; s'il a échoué, erreur avec la fin de son log
    collect_uds(job, timeout=None)   # attend la fin du job sur le cluster

for folder_base, mouse, ScoringFile in sessions:
    print(f"Processing file: {ScoringFile}")
    SleepScored = pd.read_csv(ScoringFile)
    SleepScored['label']= SleepScored['label'].str.extract(r'(\d+)', expand=False)
    SleepScored['end_time'] = SleepScored['time'] + SleepScored['duration']

    for region in ("RSC", "S1", "PFC"):
        csv = folder_base / f"UDSs_{mouse}_ml_{region}.csv"
        All_UDS = pd.read_csv(csv) if csv.exists() else pd.DataFrame()
        if len(All_UDS) == 0:
            All_UDS = pd.DataFrame([], columns = ['start time','end time',	'peak time','duration (ms)','amplitude (uV)','frequency (Hz)','quality_score','confidence','mouse_id','session_id','VigilanceState', 'toKeep', 'Overlapping_Baseline'])
        else:
            All_UDS['VigilanceState'] = All_UDS['start time'].apply(find_label)
            All_UDS['toKeep']= 'True'
            mingap=1 #in seconds
            prevDS=[]
            for ds in All_UDS.index :
                start=All_UDS.loc[ds, "start time"]
                previousend=All_UDS.loc[prevSwr, "end time"] if prevDS else start-mingap*1000
                prevDS=ds
                if start - previousend >= mingap*1000:
                    All_UDS.loc[ds, "Overlapping_Baseline"] = 'False'
                else:
                    All_UDS.loc[ds, "Overlapping_Baseline"] = 'True'
            filename = folder_base / f'UDS{region}_MLdetection.csv'
            All_UDS.to_csv(filename)
            print(len(All_UDS), f"UDS detected in {region} using ML")

    for region in ("RSC", "CA1"):
        csv = folder_base / f"SWRs_{mouse}_ml_{region}.csv"
        All_SWR = pd.read_csv(csv) if csv.exists() else pd.DataFrame()
        if len(All_SWR) == 0:
            All_SWR = pd.DataFrame([], columns = ['start time','end time',	'peak time','duration (ms)','amplitude (uV)','frequency (Hz)','quality_score','confidence','mouse_id','session_id','VigilanceState', 'toKeep', 'Overlapping_Baseline'])
        else:
            All_SWR['VigilanceState'] = All_SWR['start time'].apply(find_label)
            All_SWR['toKeep']= 'True'
            mingap=1 #in seconds
            prevSwr=[]
            for swr in All_SWR.index :
                start=All_SWR.loc[swr, "start time"]
                previousend=All_SWR.loc[prevSwr, "end time"] if prevSwr else start-mingap*1000
                prevSwr=swr
                if start - previousend >= mingap*1000:
                    All_SWR.loc[swr, "Overlapping_Baseline"] = 'False'
                else:
                    All_SWR.loc[swr, "Overlapping_Baseline"] = 'True'
        filename = folder_base / f'SWR{region}_MLdetection.csv'
        All_SWR.to_csv(filename)
        print(len(All_SWR), f"SWR detected in {region} using ML")

"""
    # Detect Global SWR in CA1 and RSC
    All_SWRCA1 = pd.read_csv(Path(f'{folder_base}/SWRCA1_MLdetection.csv'))
    All_SWRRSC = pd.read_csv(Path(f'{folder_base}/SWRRSC_MLdetection.csv'))
    All_SWRRSC['CTX']='RSC'
    All_SWRRSC['StartingLoc']='RSC'
    All_SWRRSC = All_SWRRSC.reset_index(drop=True)
    #NewPFClist=restriction_parameter(All_SWRRSC)
    NewRSClist=All_SWRRSC
    All_SWRCA1['CTX']='CA1'
    All_SWRCA1['StartingLoc']='CA1'
    All_SWRCA1 = All_SWRCA1.reset_index(drop=True)
    #NewS1list=restriction_parameter(All_SWRCA1)
    NewCA1list=All_SWRCA1
    #Spdllist=pd.concat([All_SpindlePFC,All_SWRCA1],ignore_index=False)  
    SWRlist=pd.concat([NewRSClist,NewCA1list],ignore_index=False)  
    SWRlist['LocalGlobal']='Local'
    SWRlist['DistanceClosestSWR']=np.inf
    SWRlist = SWRlist.sort_values(by='start time')
    SWRlist = SWRlist.reset_index(drop=True)
    liststarts=SWRlist["start time"]
    listends=SWRlist["end time"]
    NewSWRlist=SWRlist.copy()
    NewSWRlist['toKeep'] = NewSWRlist['toKeep'].astype(str)
    for swr1 in NewSWRlist.index : # range(len(Spdllist)) :
        start1=liststarts[swr1]
        end1=listends[swr1]
        otherunit_range = [x for x in NewSWRlist.index  if x != swr1]
        #print(f'spdl1 n°{spdl1}')
        if NewSWRlist.loc[swr1,'toKeep'] != 'False':
            for swr2 in otherunit_range:
                if NewSWRlist.loc[swr2,'toKeep'] != 'False':
                    start2=liststarts[swr2]
                    end2=listends[swr2]
                    if NewSWRlist.loc[swr1,'CTX']!= NewSWRlist.loc[swr2,'CTX']: # Needs to be from 2 differents brain areas
                        if start2>start1:
                            NewSWRlist.loc[swr1, 'DistanceClosestSWR'] = start1-start2 if start2-start1<NewSWRlist.loc[swr1, 'DistanceClosestSWR'] else NewSWRlist.loc[swr1, 'DistanceClosestSWR']
                            NewSWRlist.loc[swr2, 'DistanceClosestSWR'] = start2-start1 if start2-start1<NewSWRlist.loc[swr2, 'DistanceClosestSWR'] else NewSWRlist.loc[swr2, 'DistanceClosestSWR']
                        else:
                            NewSWRlist.loc[swr1, 'DistanceClosestSWR'] = start1-start2 if start1-start2<NewSWRlist.loc[swr1, 'DistanceClosestSWR'] else NewSWRlist.loc[swr1, 'DistanceClosestSWR']
                            NewSWRlist.loc[swr2, 'DistanceClosestSWR'] = start2-start1 if start1-start2<NewSWRlist.loc[swr2, 'DistanceClosestSWR'] else NewSWRlist.loc[swr2, 'DistanceClosestSWR']
                        if start1<=start2 and start2<=end1: # event n°2 begins after the start n°1               
                            if (end1-start2)>=int(0.5*(end1-start1)) or (end1-start2)>=int(0.5*(end2-start2)): # overlapp > to 50% of the duration of the event n°1 or the event n°2                              
                                NewSWRlist.loc[swr1, 'LocalGlobal']='Global'
                                NewSWRlist.loc[swr1, 'StartingLoc']=NewSWRlist.loc[swr1,'StartingLoc']
                                NewSWRlist.loc[swr1, 'CTX']='CA1RSC'
                                NewSWRlist.loc[swr1, 'start time']=int((start1 + start2)/2)
                                NewSWRlist.loc[swr1, 'end time']=max(end1, end2)
                                NewSWRlist.loc[swr1, 'Duration']=max(end1, end2)-int((start1 + start2)/2)
                                NewSWRlist.loc[swr2, 'toKeep']='False'
                                #print(f'Cdt n°1: Global, keep spdl n°{swr1} {start1} instead of spdl n°{swr2} {start2}')
                                break
                        elif start1<=end2 and end2<=end1: # event n°2 ends before the end n°1 
                            if (end2-start1)>=int(0.5*(end1-start1)) or (end2-start1)>=int(0.5*(end2-start2)): # overlapp > to 50% of the duration of the event n°1 or the event n°2
                                NewSWRlist.loc[swr2, 'LocalGlobal']='Global'
                                NewSWRlist.loc[swr2, 'StartingLoc']=NewSWRlist.loc[swr2,'StartingLoc']
                                NewSWRlist.loc[swr2, 'CTX']='CA1RSC'
                                NewSWRlist.loc[swr2, 'start time']=int((start1 + start2)/2)
                                NewSWRlist.loc[swr2, 'end time']=max(end1, end2)
                                NewSWRlist.loc[swr2, 'Duration']=max(end1, end2)-int((start1 + start2)/2)
                                NewSWRlist.loc[swr1, 'toKeep']='False'
                                #print(f'Cdt n°2: Global, keep spdl n°{spdl2} {start2}, instead of spdl n°{spdl1} {start1}')
                                break
    NewSWRlist = NewSWRlist[NewSWRlist['toKeep'].isin(['True', 'VRAI'])]
    NewSWRlist = NewSWRlist.sort_values(by='start time')
    NewSWRlist = NewSWRlist.reset_index(drop=True)
    NewSWRlist['DistanceClosestSWR'] = NewSWRlist['DistanceClosestSWR'] *-1 # to have in positive the spdl that arrives after the onset and vice versa
    #NewSWRlist=restriction_parameter(NewSWRlist)
    print(len(NewSWRlist[NewSWRlist['CTX']=='CA1']), 'SWR detected in CA1')
    print(len(NewSWRlist[NewSWRlist['CTX']=='RSC']), 'SWR detected in RSC')
    print(len(NewSWRlist[NewSWRlist['CTX']=='CA1RSC']), 'SWR detected in CA1RSC')
    mingap=1 #in seconds
    prevSWR=[]
    for swr in NewSWRlist.index :
        start=NewSWRlist.loc[swr, "start time"]
        previousend=NewSWRlist.loc[prevSWR, "end time"] if prevSWR else start-mingap*1000                             
        prevSWR=swr      
        if start - previousend >= mingap*1000:
            NewSWRlist.loc[swr, "Overlapping_Baseline"] = 'False'
        else:
            NewSWRlist.loc[swr, "Overlapping_Baseline"] = 'True'
    filenameOutput = folder_base / f'SWRCA1&RSC_MLdetection.csv' 
    NewSWRlist.to_csv(filenameOutput, sep= ',')
"""